In [2]:
import pandas as pd
import spacy
from spacy.language import Language
from spacy.tokens import Span
from thefuzz import fuzz, process
from tqdm.auto import tqdm
from thefuzz import fuzz, process
from NewsSentiment import TargetSentimentClassifier
import numpy as np
from typing import Literal
from functools import lru_cache
from NewsSentiment.dataset import TooLongTextException
tqdm.pandas()

In [3]:
df_articles = pd.read_csv('../Story Extraction/Opinion Pieces/Extracted/budget_reingest.csv')

df_articles

,Unnamed: 0,id,indexed_date,language,media_name,media_url,publish_date,title,url,Text
0,0,0ca24f8e5e4ee0dfa7a52c1c64f5d4d4afa6ba5abcdbf0...,2026-03-21 14:22:33.811807+00:00,en,thehindu.com,thehindu.com,2016-03-02,‘Launch of an insurrection against the Constit...,https://frontline.thehindu.com/cover-story/lau...,Several Left party leaders have studied in JNU...
1,2,41ea31bb141415c2d159188095e69a486332cb7eff88e4...,2025-09-16 18:46:49.587199+00:00,en,indiatimes.com,indiatimes.com,2013-03-21,Maruti Swift hits discount road in sales push ...,http://economictimes.indiatimes.com/news/news-...,It seems like you're already an ETPrime member...
2,3,fb80a7f207acdf0a12d2db0c495877f922f2f26d414b65...,2025-09-16 18:32:23.715725+00:00,en,thehindu.com,thehindu.com,2013-03-22,GST has a good probability in tax horizon: Par...,http://www.thehindu.com/business/Industry/gst-...,Advisor to Union Finance Minister Parthasarath...
3,5,49f1c914a7ccf3ea486c47722bbea3c89d8d53978469bc...,2025-09-16 16:37:13.564025+00:00,en,thehindu.com,thehindu.com,2013-08-06,Parliamentary supremacy under attack,http://www.thehindu.com/opinion/lead/parliamen...,Parliament is the custodian of the Constitutio...
4,6,228bb716d78e037e8c6326ade4ac23ef1baf393bd050c5...,2025-09-16 02:04:01.192949+00:00,en,indiatimes.com,indiatimes.com,2013-03-16,Nirmal Bang's five reasons why Sensex can touc...,http://economictimes.indiatimes.com/markets/an...,NEW DELHI: Increase in global liquidity and re...
...,...,...,...,...,...,...,...,...,...,...
21721,29258,0f5971bda151ce6968f3caa1bcebcf066d24b107b904f4...,2023-11-20 00:07:55+00:00,en,indiatimes.com,indiatimes.com,2023-02-16,From individuals to industries: Can green cred...,https://economictimes.indiatimes.com/prime/env...,Policy and regulations Exclusives From individ...
21722,29259,cb463eda22bcfa1c845157a00aee34b9824beb15a05307...,2023-11-16 01:42:40+00:00,en,rediff.com,rediff.com,2023-11-13,"168,491 Lives Lost In Road Mishaps",https://www.rediff.com/news/report/-168491-liv...,33 per cent of all road accidents and 35 per c...
21723,29260,646292395e078f075961bb81006a3b38cda417118efc9e...,2023-11-15 00:23:31+00:00,en,india.com,india.com,2023-11-13,PM Modi To Launch PM PVTG (Particularly Vulner...,https://www.india.com/news/india/pm-modi-to-la...,PM Modi To Launch PM PVTG (Particularly Vulner...
21724,29263,7753d0a284ba8f43b17bfa822a5a09068267fdf22f7ca9...,2023-11-12 00:15:06+00:00,en,thehindu.com,thehindu.com,2023-03-03,Programme to upskill construction workers,https://www.thehindu.com/life-and-style/homes-...,The Union Budget 2023 pushed for a focus on up...


In [4]:
df_corpus = pd.read_json('Nivaduck/nivaduck_with_display_names.json')
df_corpus = df_corpus.dropna(subset='display_name')
corpus = df_corpus['display_name'].to_dict()

In [5]:
EST = {
    'BJP',
    'AIADMK',
    'GOV',
    'NCP',
    'ABVP',
    'TDP',
    'AMMK',
    'JDU',
    'RSS',
    'JDS',
    'MNS',
    'LJP',
    'VHP'
}

OPP = {
    'INC',
    'DMK',
    'AAP',
    'SP',
    'SAMAJWADI',
    'SAMAJWADI ',
    'BJD',
    'TRS',
    'CPIM',
    'AITC',
    'TMC',
    'RJD',
    'AIMIM',
    'JKNC',
    'BSP',
    'JKPDP',
    'JMM',
    'CPIML'
}

In [6]:
@lru_cache(maxsize=2048)
def get_cached_fuzzy_match(entity_text):
    return process.extractOne(entity_text, corpus, scorer=fuzz.ratio, score_cutoff=95)

nlp = spacy.load('en_core_web_trf', disable=['tagger', 'attribute_ruler', 'lemmatizer'])
@Language.component("fuzzy_affiliation")
def fuzzy_affiliates(doc):
    ents = []
    
    for ent in doc.ents:
        if ent.label_ not in {"PERSON", "ORG"}:
            ents.append(ent)
            continue
        
        match = get_cached_fuzzy_match(ent.text)

        if not match:
            ents.append(ent)
            continue

        _, _, index =  match # pyright: ignore[reportAssignmentType]
        party = str(df_corpus.loc[index, 'party'])

        if party in EST:
            aff = 'EST'
        elif party in OPP:
            aff = 'OPP'
        else:
            ents.append(ent)
            continue
        
        new_ent = Span(doc, ent.start, ent.end, label=aff)
        ents.append(new_ent)
            
    doc.ents = ents
    return doc

nlp.add_pipe("fuzzy_affiliation", after='ner')

<function __main__.fuzzy_affiliates(doc)>

In [7]:
newsmtsc_classifier = TargetSentimentClassifier()

/opt/homebrew/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
def batch_nlp(df, entity_type:Literal['EST', 'OPP'], batch_size=16):
    #print("[Running NER...]")
    data = []
    texts = list(df['Text'])
    #for text in texts: print(text, end='\n\n')
    story_datapoints_tracker = [] # for the story at the i'th index, how many newsmtsc tuples it has
    
    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size), total=len(texts), desc="[Running NER...]"):
        
        data_count = 0

        for ent in doc.ents:
            if ent.label_ == entity_type:
                sentence = ent.sent
                left = doc.text[sentence.start_char : ent.start_char]
                entity_str = ent.text
                right= doc.text[ent.end_char : sentence.end_char]

                data.append((left, entity_str, right))
                data_count += 1
        
        story_datapoints_tracker.append(data_count)
    
    if data:
        sentiments = []
        for target in tqdm(data, desc="[Running sentiments Analysis...]"):
            try:
                s = newsmtsc_classifier.infer(targets=[target], batch_size=1)
                sentiments.extend(s)
            except TooLongTextException:
                print(f"Skipping: {target[1][:50]}...")
                sentiments.append((
                    {'class_id': 1, 'class_label': 'neutral', 'class_prob': 0.0},
                    {'class_id': 0, 'class_label': 'negative', 'class_prob': 0.0},
                    {'class_id': 2, 'class_label': 'positive', 'class_prob': 0.0}
                ))
        
        return data, story_datapoints_tracker, sentiments
    
    else:
        print("Could not find ANY relavant entities!")
        return [], [], []

def label_stories(df, entity_type:Literal['EST', 'OPP'],
                  data, story_datapoints_tracker, sentiments):
    print()
    
    label = f'{entity_type}_label'
    ent_col = f'{entity_type}_ents'
    weight_col = f'{entity_type}_weights'
    if data == []:
        df[ent_col] = pd.NA
        df[label] = 'unknown'
        df[weight_col] = pd.NA
        return df
    
    labels = []
    all_ents = []
    all_weights = []
    i = 0

    for k in story_datapoints_tracker:

        vectors = []
        weights = [0, 0, 0]
        story_ents = set()

        for datapoint, sentiment in zip(data[i:i+k],sentiments[i:i+k]):
            story_ents.add(datapoint[1])
            probs = {item['class_label']: item['class_prob'] for item in sentiment}

            if max(probs.values()) < 0.75: continue

            vector = [probs['positive'], probs['neutral'], probs['negative']]
            pred_class = np.argmax(vector)
            weights[pred_class] += 1

            print(datapoint)
            print(vector)
            print(weights)
            vectors.append(vector)
            
        
        all_ents.append(story_ents)
        all_weights.append(weights) 
        
        if not vectors:
            labels.append('unknown')
            i += k
            continue

        print()

        vectors = np.array(vectors)
        article_vec = np.mean(vectors, axis=0)
        #print('Outlet:', story['outlet'])
        #print('URL:', story['url'])
        print('Article Vector:', article_vec)
        class_index = np.argmax(article_vec)

        if class_index == 0:
            labels.append('pro')
        elif class_index == 1:
            labels.append('neutral')
        elif class_index == 2:
            labels.append('anti')
        
        i += k
        print()
        print()

    all_ents = [pd.NA if ents == set() else ents for ents in all_ents]

    df[label] = labels
    df[ent_col] = all_ents
    df[weight_col] = all_weights
    return df

#### Stance Annotation

Establishment

In [9]:
data, tracker, sents = batch_nlp(df_articles, 'EST', 16)
df_articles = label_stories(df_articles, 'EST', data, tracker, sents)

df_articles

Processing batches:   0%|          | 0/1 [00:00<?, ?batch/s]


Skipping: NITI Aayog...


Processing batches:   0%|          | 0/1 [00:00<?, ?batch/s]


Skipping: Sanjay Gupta...


[Running sentiments Analysis...]: 100%|██████████| 16806/16806 [1:20:22<00:00,  3.48it/s]



('Everything, however, would depend on execution, and, if implemented, there could be an upside too of up to 6.5 per cent, Managing Director & CEO ', 'Atul Joshi', ' said, adding the rating and research agency expected the 2013-14 GDP (gross domestic product) growth to be 6.1 per cent.')
[0.17551758885383606, 0.8130015730857849, 0.011480854824185371]
[0, 1, 0]

Article Vector: [0.17551759 0.81300157 0.01148085]


('Backing the draft report of the Standing Committee, ', 'BJP', ' spokesman Prakash Javadekar told The Hindu that the party’s position was the same as that articulated by Mr. Sinha.')
[0.0271591953933239, 0.9307425022125244, 0.042098335921764374]
[0, 1, 0]
('Backing the draft report of the Standing Committee, BJP spokesman ', 'Prakash Javadekar', ' told The Hindu that the party’s position was the same as that articulated by Mr. Sinha.')
[0.018921956419944763, 0.972096860408783, 0.008981170132756233]
[0, 2, 0]

Article Vector: [0.02304058 0.95141968 0.02553975]


('', 'Bharati

,Unnamed: 0,id,indexed_date,language,media_name,media_url,publish_date,title,url,Text,EST_label,EST_ents,EST_weights
0,0,0ca24f8e5e4ee0dfa7a52c1c64f5d4d4afa6ba5abcdbf0...,2026-03-21 14:22:33.811807+00:00,en,thehindu.com,thehindu.com,2016-03-02,‘Launch of an insurrection against the Constit...,https://frontline.thehindu.com/cover-story/lau...,Several Left party leaders have studied in JNU...,unknown,{Bharatiya Janata Party},"[0, 0, 0]"
1,2,41ea31bb141415c2d159188095e69a486332cb7eff88e4...,2025-09-16 18:46:49.587199+00:00,en,indiatimes.com,indiatimes.com,2013-03-21,Maruti Swift hits discount road in sales push ...,http://economictimes.indiatimes.com/news/news-...,It seems like you're already an ETPrime member...,unknown,<NA>,"[0, 0, 0]"
2,3,fb80a7f207acdf0a12d2db0c495877f922f2f26d414b65...,2025-09-16 18:32:23.715725+00:00,en,thehindu.com,thehindu.com,2013-03-22,GST has a good probability in tax horizon: Par...,http://www.thehindu.com/business/Industry/gst-...,Advisor to Union Finance Minister Parthasarath...,unknown,<NA>,"[0, 0, 0]"
3,5,49f1c914a7ccf3ea486c47722bbea3c89d8d53978469bc...,2025-09-16 16:37:13.564025+00:00,en,thehindu.com,thehindu.com,2013-08-06,Parliamentary supremacy under attack,http://www.thehindu.com/opinion/lead/parliamen...,Parliament is the custodian of the Constitutio...,unknown,<NA>,"[0, 0, 0]"
4,6,228bb716d78e037e8c6326ade4ac23ef1baf393bd050c5...,2025-09-16 02:04:01.192949+00:00,en,indiatimes.com,indiatimes.com,2013-03-16,Nirmal Bang's five reasons why Sensex can touc...,http://economictimes.indiatimes.com/markets/an...,NEW DELHI: Increase in global liquidity and re...,unknown,<NA>,"[0, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21721,29258,0f5971bda151ce6968f3caa1bcebcf066d24b107b904f4...,2023-11-20 00:07:55+00:00,en,indiatimes.com,indiatimes.com,2023-02-16,From individuals to industries: Can green cred...,https://economictimes.indiatimes.com/prime/env...,Policy and regulations Exclusives From individ...,unknown,<NA>,"[0, 0, 0]"
21722,29259,cb463eda22bcfa1c845157a00aee34b9824beb15a05307...,2023-11-16 01:42:40+00:00,en,rediff.com,rediff.com,2023-11-13,"168,491 Lives Lost In Road Mishaps",https://www.rediff.com/news/report/-168491-liv...,33 per cent of all road accidents and 35 per c...,unknown,<NA>,"[0, 0, 0]"
21723,29260,646292395e078f075961bb81006a3b38cda417118efc9e...,2023-11-15 00:23:31+00:00,en,india.com,india.com,2023-11-13,PM Modi To Launch PM PVTG (Particularly Vulner...,https://www.india.com/news/india/pm-modi-to-la...,PM Modi To Launch PM PVTG (Particularly Vulner...,pro,{Narendra Modi},"[1, 0, 0]"
21724,29263,7753d0a284ba8f43b17bfa822a5a09068267fdf22f7ca9...,2023-11-12 00:15:06+00:00,en,thehindu.com,thehindu.com,2023-03-03,Programme to upskill construction workers,https://www.thehindu.com/life-and-style/homes-...,The Union Budget 2023 pushed for a focus on up...,unknown,<NA>,"[0, 0, 0]"


Opposition

In [10]:
data, tracker, sents = batch_nlp(df_articles, 'OPP')
df_articles = label_stories(df_articles, 'OPP', data, tracker, sents)
df_articles.to_parquet('budget_reingest_annotated.parquet', index=False)

df_articles

[Running sentiments Analysis...]: 100%|██████████| 5097/5097 [17:39<00:00,  4.81it/s]



('However, Finance Minister ', 'P Chidambaram', ' reassured investors of higher growth and hoped that GDP will grow by over 6 per cent in the next financial year.')
[0.9183873534202576, 0.06952022761106491, 0.012092378921806812]
[1, 0, 0]

Article Vector: [0.91838735 0.06952023 0.01209238]


('Union Finance Minister ', 'P Chidambaram', ' on Saturday said his budget proposal on setting up the country’s first all-women bank has received good all-round response.')
[0.7946118712425232, 0.19454145431518555, 0.010846606455743313]
[1, 0, 0]

Article Vector: [0.79461187 0.19454145 0.01084661]


('', 'P. Chidambaram', ' has upped the ante by referring to the “laws of economics” implying that they cannot be bucked and suggesting that his own actions reflect an awareness of them.')
[0.9296721816062927, 0.04091404005885124, 0.029413754120469093]
[1, 0, 0]

Article Vector: [0.92967218 0.04091404 0.02941375]


('When Union Finance Minister ', 'P. Chidambaram', ' announced extension of the Rashtriya

,Unnamed: 0,id,indexed_date,language,media_name,media_url,publish_date,title,url,Text,EST_label,EST_ents,EST_weights,OPP_label,OPP_ents,OPP_weights
0,0,0ca24f8e5e4ee0dfa7a52c1c64f5d4d4afa6ba5abcdbf0...,2026-03-21 14:22:33.811807+00:00,en,thehindu.com,thehindu.com,2016-03-02,‘Launch of an insurrection against the Constit...,https://frontline.thehindu.com/cover-story/lau...,Several Left party leaders have studied in JNU...,unknown,{Bharatiya Janata Party},"[0, 0, 0]",unknown,{Sitaram Yechury},"[0, 0, 0]"
1,2,41ea31bb141415c2d159188095e69a486332cb7eff88e4...,2025-09-16 18:46:49.587199+00:00,en,indiatimes.com,indiatimes.com,2013-03-21,Maruti Swift hits discount road in sales push ...,http://economictimes.indiatimes.com/news/news-...,It seems like you're already an ETPrime member...,unknown,<NA>,"[0, 0, 0]",unknown,<NA>,"[0, 0, 0]"
2,3,fb80a7f207acdf0a12d2db0c495877f922f2f26d414b65...,2025-09-16 18:32:23.715725+00:00,en,thehindu.com,thehindu.com,2013-03-22,GST has a good probability in tax horizon: Par...,http://www.thehindu.com/business/Industry/gst-...,Advisor to Union Finance Minister Parthasarath...,unknown,<NA>,"[0, 0, 0]",unknown,<NA>,"[0, 0, 0]"
3,5,49f1c914a7ccf3ea486c47722bbea3c89d8d53978469bc...,2025-09-16 16:37:13.564025+00:00,en,thehindu.com,thehindu.com,2013-08-06,Parliamentary supremacy under attack,http://www.thehindu.com/opinion/lead/parliamen...,Parliament is the custodian of the Constitutio...,unknown,<NA>,"[0, 0, 0]",unknown,<NA>,"[0, 0, 0]"
4,6,228bb716d78e037e8c6326ade4ac23ef1baf393bd050c5...,2025-09-16 02:04:01.192949+00:00,en,indiatimes.com,indiatimes.com,2013-03-16,Nirmal Bang's five reasons why Sensex can touc...,http://economictimes.indiatimes.com/markets/an...,NEW DELHI: Increase in global liquidity and re...,unknown,<NA>,"[0, 0, 0]",pro,{P Chidambaram},"[1, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21721,29258,0f5971bda151ce6968f3caa1bcebcf066d24b107b904f4...,2023-11-20 00:07:55+00:00,en,indiatimes.com,indiatimes.com,2023-02-16,From individuals to industries: Can green cred...,https://economictimes.indiatimes.com/prime/env...,Policy and regulations Exclusives From individ...,unknown,<NA>,"[0, 0, 0]",unknown,<NA>,"[0, 0, 0]"
21722,29259,cb463eda22bcfa1c845157a00aee34b9824beb15a05307...,2023-11-16 01:42:40+00:00,en,rediff.com,rediff.com,2023-11-13,"168,491 Lives Lost In Road Mishaps",https://www.rediff.com/news/report/-168491-liv...,33 per cent of all road accidents and 35 per c...,unknown,<NA>,"[0, 0, 0]",unknown,<NA>,"[0, 0, 0]"
21723,29260,646292395e078f075961bb81006a3b38cda417118efc9e...,2023-11-15 00:23:31+00:00,en,india.com,india.com,2023-11-13,PM Modi To Launch PM PVTG (Particularly Vulner...,https://www.india.com/news/india/pm-modi-to-la...,PM Modi To Launch PM PVTG (Particularly Vulner...,pro,{Narendra Modi},"[1, 0, 0]",unknown,<NA>,"[0, 0, 0]"
21724,29263,7753d0a284ba8f43b17bfa822a5a09068267fdf22f7ca9...,2023-11-12 00:15:06+00:00,en,thehindu.com,thehindu.com,2023-03-03,Programme to upskill construction workers,https://www.thehindu.com/life-and-style/homes-...,The Union Budget 2023 pushed for a focus on up...,unknown,<NA>,"[0, 0, 0]",unknown,<NA>,"[0, 0, 0]"
